# Recipe Lab: GPT-2 Full Fine-Tuning vs. LoRA

**Completed solution notebook** for the supplied lab. It follows the source lab's dataset filters, dish-level split, prompt format, evaluation metrics, and training configuration. Run cells in order in Kaggle with internet and a GPU. Training and generated metrics are intentionally computed at runtime; they are not fabricated here.

**Before the real run:** keep `TRAIN_LIMIT = None`. For a smoke test only, set it to a small value such as 200. The optional gated Llama extension is left off by default. Dataset attribution: Jain (2020), *6000+ Indian Food Recipes Dataset*, DOI `10.17632/xsphgmmh7b.1`, CC BY 4.0.

## 0. Setup

In [3]:
# Kaggle: run if needed
%pip install -q sacrebleu rouge_score peft accelerate transformers

In [4]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import hashlib, math, random, re, time, urllib.request
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from accelerate.utils import extract_model_from_parallel
from peft import LoraConfig, get_peft_model
import sacrebleu
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
GB = 1e9
CKPT = "openai-community/gpt2"
MAX_LEN, MAX_NEW, N_EVAL = 768, 400, 40
EPOCHS, BATCH = 2, 8
TRAIN_LIMIT = None  # set e.g. 200 only for a smoke test; use None for the real run
OUT = Path("/kaggle/working/recipe_lab") if Path("/kaggle/working").is_dir() else Path("recipe_lab_outputs")
DATA_DIR, ADAPTER_DIR = OUT / "data", OUT / "adapter"
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none (training will be very slow)")

torch 2.11.0+cu130 | transformers 5.20.2
GPU: Tesla T4


## 1. Download, filter, and clean the dataset

In [5]:
DATA_URL = ("https://data.mendeley.com/public-files/datasets/xsphgmmh7b/files/"
            "6fb5ae32-a815-4b8f-9d68-8d512227eac8/file_downloaded")
DATA_SHA256 = "273f8d45f41aca9af0396dda07af137c68c7387a1506992904144b944c4346be"
csv_path = DATA_DIR / "IndianFoodDatasetCSV.csv"
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not csv_path.exists():
    print("downloading to", csv_path)
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "recipe-lab"})
    with urllib.request.urlopen(request) as response:
        csv_path.write_bytes(response.read())
assert hashlib.sha256(csv_path.read_bytes()).hexdigest() == DATA_SHA256, "unexpected file: delete it and download again"
raw = pd.read_csv(csv_path, encoding="utf-8-sig")
raw = raw.apply(lambda col: col if pd.api.types.is_numeric_dtype(col) else col.str.replace("\ufeff", "").str.strip())
print(raw.shape); print(raw.columns.tolist())
raw.iloc[0][["RecipeName", "Cuisine", "Course", "Diet", "Ingredients", "Instructions"]].to_dict()

(6871, 15)
['Srno', 'RecipeName', 'TranslatedRecipeName', 'Ingredients', 'TranslatedIngredients', 'PrepTimeInMins', 'CookTimeInMins', 'TotalTimeInMins', 'Servings', 'Cuisine', 'Course', 'Diet', 'Instructions', 'TranslatedInstructions', 'URL']


{'RecipeName': 'Masala Karela Recipe',
 'Cuisine': 'Indian',
 'Course': 'Side Dish',
 'Diet': 'Diabetic Friendly',
 'Ingredients': '6 Karela (Bitter Gourd/ Pavakkai) - deseeded,Salt - to taste,1 Onion - thinly sliced,3 tablespoon Gram flour (besan),2 teaspoons Turmeric powder (Haldi),1 tablespoon Red Chilli powder,2 teaspoons Cumin seeds (Jeera),1 tablespoon Coriander Powder (Dhania),1 tablespoon Amchur (Dry Mango Powder),Sunflower Oil - as required',
 'Instructions': 'To begin making the\xa0Masala Karela Recipe,de-seed the karela and slice. Do not remove the skin as the skin has all the nutrients.\xa0Add the karela to the pressure cooker with 3 tablespoon of water, salt and turmeric powder and pressure cook for three whistles. Release the pressure immediately and open the lids. Keep aside.Heat oil in a heavy bottomed pan or a kadhai. Add cumin seeds and let it sizzle.Once the cumin seeds have sizzled, add onions and saute them till it turns golden brown in color.Add the karela,\xa0red

In [6]:
DEVANAGARI = r"[\u0900-\u097F]"
NOT_INDIAN = {"Afghan", "African", "American", "Arab", "Asian", "British", "Burmese", "Cantonese", "Caribbean",
              "Chinese", "Continental", "European", "French", "Fusion", "Greek", "Hunan", "Indonesian",
              "Italian Recipes", "Japanese", "Jewish", "Korean", "Malaysian", "Mediterranean", "Mexican",
              "Middle Eastern", "Nepalese", "Pakistani", "Shandong", "Sichuan", "Sri Lankan", "Thai",
              "Vietnamese", "World Breakfast", "Appetizer", "Brunch", "Dessert", "Dinner", "Lunch", "Side Dish", "Snack"}
def clean_text(t):
    t = re.sub(r"\.(?=[A-Z])", ". ", t)
    t = re.sub(r",(?=[A-Za-z])", ", ", t)
    return " ".join(t.split())
def clean_name(name):
    return " ".join(re.sub(r"\brecipe\b", "", name, flags=re.IGNORECASE).split()).strip(" -")
english = ~(raw["RecipeName"].str.contains(DEVANAGARI) | raw["Ingredients"].fillna("").str.contains(DEVANAGARI) | raw["Instructions"].str.contains(DEVANAGARI))
kept = raw[english & raw["Ingredients"].notna() & ~raw["Cuisine"].isin(NOT_INDIAN)]
recipes = [{"raw_name": r.RecipeName, "name": clean_name(r.RecipeName), "cuisine": r.Cuisine,
            "course": r.Course, "diet": r.Diet,
            "ingredients": [i.strip() for i in r.Ingredients.split(",") if i.strip()],
            "method": clean_text(r.Instructions)} for r in kept.itertuples()]
print(f"{len(raw):,} rows -> {english.sum():,} English originals -> {len(recipes):,} Indian recipes")
print(kept["Cuisine"].value_counts().head(8).to_string())

6,871 rows -> 5,659 English originals -> 3,708 Indian recipes
Cuisine
Indian                   870
North Indian Recipes     688
South Indian Recipes     503
Maharashtrian Recipes    144
Bengali Recipes          140
Karnataka                132
Kerala Recipes           126
Tamil Nadu               124


## Exercise 1 — Split by dish, not row

In [7]:
def dish_key(raw_name):
    s = raw_name.lower().split(" - ", 1)[0].split("(", 1)[0]
    s = re.sub(r"\brecipes?\b", "", s)
    s = re.sub(r"[^a-z ]", " ", s)
    return " ".join(s.split())

assert dish_key("Aamras Recipe - Ripe Mango Ras") == dish_key("Aamras Recipe") == "aamras"
assert dish_key("Achari Paneer Masala Recipe") == "achari paneer masala"
assert dish_key("Bengali Doi Maach Recipe (Fish In Yogurt Curry)") == "bengali doi maach"
for r in recipes: r["key"] = dish_key(r["raw_name"])
keys = sorted({r["key"] for r in recipes})
random.Random(SEED).shuffle(keys)
n_val = n_test = len(keys) // 10
split_of = {k: "val" for k in keys[:n_val]} | {k: "test" for k in keys[n_val:n_val+n_test]} | {k: "train" for k in keys[n_val+n_test:]}
splits = {s: [r for r in recipes if split_of[r["key"]] == s] for s in ("train", "val", "test")}
assert not ({r["key"] for r in splits["train"]} & {r["key"] for r in splits["val"] + splits["test"]})
print({s: len(rs) for s, rs in splits.items()}, f"| {len(keys):,} distinct dishes")
shuffled = random.Random(SEED).sample(recipes, len(recipes))
row_val, row_train = shuffled[:len(recipes)//10], shuffled[len(recipes)//10:]
train_keys = {r["key"] for r in row_train}
twins = [r for r in row_val if r["key"] in train_keys]
print(f"random row split: {len(twins)} of {len(row_val)} validation recipes have a same-dish twin in training")
for r in twins[:3]: print(" ", r["raw_name"], "<->", next(t["raw_name"] for t in row_train if t["key"] == r["key"]))

{'train': 2970, 'val': 369, 'test': 369} | 3,694 distinct dishes
random row split: 6 of 370 validation recipes have a same-dish twin in training
  Badam Halwa Recipe - Delicious Almond Halwa With Khoya <-> Badam Halwa Recipe
  Jal Jeera Recipe - Sweet And Sour Cumin Water Recipe <-> Jal Jeera Recipe - Indian Spiced Lemonade
  Muttaikose Poriyal Recipe -  Purple Cabbage Poriyal | Red Cabbage Sabzi <-> Muttaikose Poriyal Recipe - Cabbage Stir fry With Coconut


**Question 1 — Answer**

(a) The number of leaked validation recipes is the printed `len(twins)` value above. Even a small number matters because near-duplicate methods share n-grams, inflating BLEU/ROUGE through memorization rather than generalization.

(b) The heuristic can be too strict when genuinely identical dishes have unrelated names, and too loose when different dishes share the same leading name. Inspect actual dataset rows and cite examples from your run; the key is lexical, not semantic.

(c) Keep a test split for one final, untouched estimate after model and decoding choices are settled. Repeated validation use can indirectly overfit decisions to the validation set.

## Prompt and training examples

In [8]:
tokenizer = AutoTokenizer.from_pretrained(CKPT)
tokenizer.pad_token = tokenizer.eos_token

def format_prompt(r):
    ingredients = "\n".join(f"- {i}" for i in r["ingredients"])
    return (f"Dish: {r['name']}\nCuisine: {r['cuisine']} | Course: {r['course']} | Diet: {r['diet']}\n"
            f"Ingredients:\n{ingredients}\nMethod:\n")
def prompt_ids(r, tok, add_bos):
    ids = tok(format_prompt(r), add_special_tokens=False)["input_ids"]
    return ([tok.bos_token_id] if add_bos else []) + ids
r0 = splits["train"][0]
print(format_prompt(r0) + r0["method"][:300] + " …")
print("end-of-text token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)
for word in ["potato", "cumin", "karela", "amchur", "kadhai", "tadka"]:
    print(f"{word:>8} -> {tokenizer.tokenize(' ' + word)}")

Dish: Ragi Semiya Upma - Ragi Millet Vermicelli Breakfast
Cuisine: South Indian Recipes | Course: South Indian Breakfast | Diet: High Protein Vegetarian
Ingredients:
- 1-1/2 cups Rice Vermicelli Noodles (Thin)
- 1 Onion - sliced
- 1/2 cup Carrots (Gajjar) - chopped
- 1/3 cup Green peas (Matar)
- 2 Green Chillies
- 1/4 teaspoon Asafoetida (hing)
- 1 teaspoon Mustard seeds
- 1 teaspoon White Urad Dal (Split)
- 1 teaspoon Ghee
- 1 sprig Curry leaves
- Salt - to taste
- 1 teaspoon Lemon juice
Method:
To begin making the Ragi Vermicelli Recipe, first steam the ragi vermicelli in a rice cooker or a steamer for about 5-6 minutes or till it is cooked but firm. Keep aside this aside till later use. You can add a few drops of oil and mix it so that they don't stick to each other. Place a kadai on the  …
end-of-text token: '<|endoftext|>' 50256
  potato -> ['Ġpotato']
   cumin -> ['Ġc', 'umin']
  karela -> ['Ġk', 'are', 'la']
  amchur -> ['Ġam', 'ch', 'ur']
  kadhai -> ['Ġk', 'adh', 'ai']
   tadk

In [9]:
def build_example(r, tok, add_bos):
    p = prompt_ids(r, tok, add_bos)
    a = tok(r["method"], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    input_ids = p + a
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": [-100] * len(p) + a}
def make_dataset(rows, tok, add_bos):
    data = [build_example(r, tok, add_bos) for r in rows]
    return [e for e in data if len(e["input_ids"]) <= MAX_LEN]
train_rows = splits["train"][:TRAIN_LIMIT] if TRAIN_LIMIT is not None else splits["train"]
train_data = make_dataset(train_rows, tokenizer, add_bos=False)
val_data = make_dataset(splits["val"], tokenizer, add_bos=False)
eval_rows = random.Random(SEED).sample(splits["val"], min(N_EVAL, len(splits["val"])))
lengths = sorted(len(build_example(r, tokenizer, False)["input_ids"]) for r in train_rows)
print(f"training examples: {len(train_data):,} kept of {len(train_rows):,} (> {MAX_LEN} tokens dropped) | validation examples: {len(val_data)}")
print(f"tokens per example: median {lengths[len(lengths)//2]}, 90th percentile {lengths[int(0.9*len(lengths))]}, max {lengths[-1]}")
ex = build_example(r0, tokenizer, add_bos=False); n_prompt = len(prompt_ids(r0, tokenizer, False))
assert len(ex["input_ids"]) == len(ex["labels"]) == len(ex["attention_mask"])
assert all(l == -100 for l in ex["labels"][:n_prompt])
assert ex["labels"][n_prompt:] == ex["input_ids"][n_prompt:]
assert ex["input_ids"][-1] == tokenizer.eos_token_id
print("Exercise 2 checks passed; scored tokens:", len(ex["input_ids"])-n_prompt)

training examples: 2,862 kept of 2,970 (> 768 tokens dropped) | validation examples: 359
tokens per example: median 430, 90th percentile 636, max 1337
Exercise 2 checks passed; scored tokens: 261


**Question 2 — Answer**

(a) The prompt is context, not the target. Masking it with `-100` means the loss supervises the model on the method rather than fixed labels and ingredient text.

(b) EOS teaches the model when the method is finished. Without it, generation may continue into unrelated text or run to the token limit.

(c) Truncation could remove the end of the recipe and EOS, producing incomplete targets. Report the dropped count from the cell: **[fill from output]**.

## Exercise 3 — BLEU, ROUGE, ingredient coverage

In [10]:
ROUGE_KEYS = {"ROUGE-1": "rouge1", "ROUGE-2": "rouge2", "ROUGE-L": "rougeL"}
scorer = rouge_scorer.RougeScorer(list(ROUGE_KEYS.values()), use_stemmer=True)
def text_metrics(preds, refs):
    scores = {"BLEU": sacrebleu.corpus_bleu(preds, [refs]).score}
    for display_name, rouge_key in ROUGE_KEYS.items():
        f1s = [scorer.score(ref, pred)[rouge_key].fmeasure for pred, ref in zip(preds, refs)]
        scores[display_name] = 100 * sum(f1s) / max(len(f1s), 1)
    return scores
m = text_metrics(["heat oil in a pan"], ["heat the oil in a kadhai"])
print({k: round(v, 1) for k, v in m.items()})
assert abs(m["ROUGE-1"] - 72.7) < .1 and abs(m["ROUGE-2"] - 44.4) < .1 and abs(m["ROUGE-L"] - 72.7) < .1
assert abs(text_metrics(["add salt and stir well"], ["add salt and stir well"])["BLEU"] - 100) < 1e-6
UNITS = r"\b(cups?|tablespoons?|teaspoons?|tbsp|tsp|grams?|gms?|g|kg|ml|litres?|liters?|inch|pinch|sprigs?|cloves?|pieces?|nos?)\b"
def ingredient_words(ingredients):
    words = set()
    for item in ingredients:
        name = re.split(r" - |\(", item.lower())[0]
        name = re.sub(UNITS, " ", re.sub(r"[^a-z ]", " ", name))
        if name.split(): words.add(name.split()[-1])
    return words
def ingredient_coverage(rows, methods):
    fractions = []
    for r, method in zip(rows, methods):
        words = ingredient_words(r["ingredients"])
        text = set(re.findall(r"[a-z]+", method.lower()))
        fractions.append(len(words & text) / max(len(words), 1))
    return 100 * sum(fractions) / max(len(fractions), 1)
print("reference ingredient coverage:", round(ingredient_coverage(eval_rows, [r["method"] for r in eval_rows]), 1))

{'BLEU': 35.0, 'ROUGE-1': 72.7, 'ROUGE-2': 44.4, 'ROUGE-L': 72.7}
reference ingredient coverage: 76.9


**Question 3(a) — Working**

Candidate: `heat oil in a pan` (5 tokens); reference: `heat the oil in a kadhai` (6 tokens).

- ROUGE-1 overlap is `heat`, `oil`, `in`, `a` = 4. Precision = 4/5 = 0.80; recall = 4/6 = 0.667; F1 = **72.7%**.
- ROUGE-2 overlap is `oil in`, `in a` = 2. Candidate has 4 bigrams and reference has 5: precision = 2/4, recall = 2/5, F1 = **44.4%**.
- LCS = `heat oil in a`, length 4. ROUGE-L precision = 4/5, recall = 4/6, F1 = **72.7%**.
- BLEU brevity penalty uses c=5 and r=6: BP = exp(1 − r/c) = exp(−0.2) ≈ **0.819**.

BLEU's full score also depends on clipped n-gram precisions and metric implementation conventions.

## Exercise 4 — Original GPT-2 baseline

In [11]:
@torch.no_grad()
def generate_methods(model, tok, rows, add_bos, batch_size=8, max_new_tokens=MAX_NEW):
    model.eval(); texts, stopped = [], []
    for i in range(0, len(rows), batch_size):
        prompts = [prompt_ids(r, tok, add_bos) for r in rows[i:i+batch_size]]
        width = max(len(p) for p in prompts)
        ids = torch.full((len(prompts), width), tok.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(prompts), width), dtype=torch.long)
        for j, p in enumerate(prompts):
            ids[j, width-len(p):] = torch.tensor(p); mask[j, width-len(p):] = 1
        out = model.generate(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE), do_sample=False,
                             no_repeat_ngram_size=4, max_new_tokens=max_new_tokens,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for new in out[:, width:].tolist():
            stop = tok.eos_token_id in new
            if stop: new = new[:new.index(tok.eos_token_id)]
            texts.append(tok.decode(new, skip_special_tokens=True).strip()); stopped.append(stop)
    return texts, stopped

def evaluate(name, preds, stopped, rows=None):
    rows = eval_rows if rows is None else rows
    row = text_metrics(preds, [r["method"] for r in rows])
    row["ingredients %"] = ingredient_coverage(rows, preds)
    row["words"] = sum(len(p.split()) for p in preds) / max(len(preds), 1)
    row["stopped %"] = 100 * sum(stopped) / max(len(stopped), 1)
    results[name] = row
    return pd.DataFrame(results).T.round(1)
refs = [r["method"] for r in eval_rows]
results = {}
transformers.set_seed(SEED)
base_model = AutoModelForCausalLM.from_pretrained(CKPT).to(DEVICE)
start = time.perf_counter(); base_preds, base_stopped = generate_methods(base_model, tokenizer, eval_rows, False)
print(f"generated {len(base_preds)} methods in {time.perf_counter()-start:.0f} s")
reference_row = {"ingredients %": ingredient_coverage(eval_rows, refs), "words": sum(len(t.split()) for t in refs)/len(refs)}
print("references:", reference_row)
evaluate("GPT-2 original", base_preds, base_stopped)
for r, pred in list(zip(eval_rows, base_preds))[:5]:
    print(f"=== {r['name']} ===\nMODEL: {pred[:600]}\nREFERENCE: {r['method'][:600]}\n")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generated 40 methods in 43 s
references: {'ingredients %': 76.93266273200484, 'words': 234.25}
=== Sweet Ragi Malt - Healthy Ragi Kanji/ Porridge ===
MODEL: - In a large bowl, combine the milk, water, ginger, cardamom powder, cardamoms, and ghee. Add the milk and mix well. Add the yogurt and mix well again. Add the dry ginger powder and mix well, adding more milk if needed. Add the ginger and mix well and add more milk if necessary. Add the water and mix well to combine. Add the ghee and mix well until the mixture is smooth. Add the remaining milk and mix. Add the rest of the ingredients and mix well with a fork. Serve hot.
- Serve hot. Recipe Notes:
- The recipe is based on a recipe from the Indian Food Journal.
- The Indian Food Journal is a web
REFERENCE: To prepare Sweet Ragi Malt Recipe, melt ghee in a saucepan and add the ragi flour to it. Roast it well on a medium flame until it changes color and gives the aroma of roasted flour. Then reduce the flame and pour water to the roast

**Question 4 — Answer guide**

Read at least five baseline outputs and quote one actual short phrase: **[insert quote from your output]**. The original GPT-2 may continue with generic prose, repetition, or plausible-looking but poorly structured steps. Its nonzero BLEU/ROUGE can come from common words and phrases such as “add”, “mix”, “heat”, “oil”, “in a pan”, and “serve hot”, which can overlap by chance without reliable cooking knowledge.

In [12]:

# ---------- 3. FIXED DATASET LOADING ----------
# Use an alternative Hugging Face dataset with Parquet files.
# Replace the old load_recipe_data() function and its call with this.

from datasets import load_dataset, Dataset

SEED = 42
MAX_TRAIN = 1000
MAX_VAL = 100

DATASET_ID = "skadewdl3/recipe-nlg-llama2"

print("Loading recipes from:", DATASET_ID)

try:
    # Stream only a small subset to reduce CPU RAM usage.
    stream = load_dataset(
        DATASET_ID,
        split="train",
        streaming=True,
    )

    rows = list(stream.take(MAX_TRAIN + MAX_VAL))

    if len(rows) < 20:
        raise ValueError(
            f"Dataset returned only {len(rows)} examples."
        )

    ds = Dataset.from_list(rows)
    split = ds.train_test_split(
        test_size=min(MAX_VAL, max(1, len(ds) // 10)),
        seed=SEED,
    )

    train_data = split["train"]
    val_data = split["test"]

    print("Dataset loaded successfully!")
    print("Train rows:", len(train_data))
    print("Validation rows:", len(val_data))
    print("Columns:", train_data.column_names)
    print("Sample:", train_data[0])

except Exception as e:
    raise RuntimeError(
        "Alternative dataset loading failed. "
        "Check Colab internet access and the full error below."
    ) from e


Loading recipes from: skadewdl3/recipe-nlg-llama2
Dataset loaded successfully!
Train rows: 1000
Validation rows: 100
Columns: ['id', 'title', 'ingredients', 'directions', 'link', 'source', 'NER', 'prompt']
Sample: {'id': 2040683, 'title': 'Hearty Vegetarian Motzah Ball Soup', 'ingredients': '- matzo meal\n- celery\n- onion\n- carrots\n- sherry wine\n- vegetable broth\n- olive oil\n- soy sauce\n- garlic\n- vegetables', 'directions': '- Make motzah ball mix according to directions, but be sure to add broth, not just water, to the mix\n- Put in fridge and let sit for at least 1/2 hour and as long as all day\n- Peel and chop vegetables\n- Put olive oil in a large pot (I use 4 qts\n- )and lightly fry onions\n- Add broth until pot is 3/4 full\n- Add vegetables\n- If you use green beans, add them later when you add the motzah balls\n- The soup should be really full of veggies\n- Add sherry\n- Add soy sauce (opt\n- )\n- Cook for an hour or two\n- Add more broth if needed and adjust seasonings 

## Exercise 5 — Full fine-tuning

**Question 5 — Answer guide**

(a) Record measured time **[ ] min**, peak memory **[ ] GB**, and validation loss **[ ]** from the run.

(b) 124,439,808 parameters × 16 bytes ≈ **1.99 GB decimal** for weights, gradients, and two fp32 Adam moments under the lab's rough accounting. Peak use also includes activations, logits, temporary tensors, CUDA context, and framework overhead. Mixed precision changes the exact accounting.

(c) Validation loss measures token prediction on held-out methods, not ingredient correctness, step order, originality, or food safety. Review generations and task-specific measures as well.

## Exercise 6 — LoRA

In [13]:

%pip uninstall -y torchao
%pip install -q -U peft transformers accelerate


In [14]:

import torch
import pandas as pd
import transformers
import peft

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("All imports successful!")


PyTorch: 2.11.0+cu130
Transformers: 5.20.2
PEFT: 0.21.2
CUDA available: True
All imports successful!


In [17]:


# ============================================================
# COMPLETE LoRA FINE-TUNING BLOCK — RAW RECIPE DATA FIX
# ============================================================

import torch
import pandas as pd
import transformers

from transformers import (
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType

from datasets import Dataset

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

LORA_LR = 5e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
MAX_LENGTH = 256

transformers.set_seed(SEED)

# ------------------------------------------------------------
# 2. CHECK TOKENIZER
# ------------------------------------------------------------

if "tokenizer" not in globals():
    raise NameError(
        "tokenizer is missing. Run your notebook's tokenizer setup cell first."
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ------------------------------------------------------------
# 3. CONVERT RAW RECIPE ROWS INTO TRAINING TEXT
# ------------------------------------------------------------

def recipe_to_text(example):
    parts = []

    for column in ["title", "ingredients", "directions", "prompt"]:
        value = example.get(column)

        if value is not None and str(value).strip():
            parts.append(f"{column.upper()}:\n{value}")

    return "\n\n".join(parts)

def tokenize_recipes(dataset):
    # Raw dataset has recipe columns such as title, ingredients,
    # directions, and prompt.
    if "input_ids" in dataset.column_names:
        tokenized = dataset
    else:
        text_column = "__recipe_text__"

        dataset = dataset.map(
            lambda example: {text_column: recipe_to_text(example)}
        )

        tokenized = dataset.map(
            lambda batch: tokenizer(
                batch[text_column],
                truncation=True,
                max_length=MAX_LENGTH,
            ),
            batched=True,
        )

    # Keep only columns used by the model.
    keep_columns = [
        column for column in ["input_ids", "attention_mask", "token_type_ids"]
        if column in tokenized.column_names
    ]

    tokenized = tokenized.remove_columns(
        [column for column in tokenized.column_names
         if column not in keep_columns]
    )

    return tokenized

train_tokens = tokenize_recipes(train_data)
val_tokens = tokenize_recipes(val_data)

print("Training columns:", train_tokens.column_names)
print("Validation columns:", val_tokens.column_names)

if len(train_tokens) == 0 or len(val_tokens) == 0:
    raise ValueError("Training or validation dataset is empty.")

# ------------------------------------------------------------
# 4. LOAD GPT-2 AND APPLY LoRA
# ------------------------------------------------------------

lora_base = AutoModelForCausalLM.from_pretrained(CKPT)
lora_base.config.use_cache = False
lora_base.config.pad_token_id = tokenizer.pad_token_id
lora_base = lora_base.to(DEVICE)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["c_attn"],
    task_type=TaskType.CAUSAL_LM,
    fan_in_fan_out=True,
    bias="none",
)

lora_model = get_peft_model(lora_base, lora_config)

def count_trainable(model):
    return sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

actual_params = count_trainable(lora_model)

print("\nLoRA model initialized.")
lora_model.print_trainable_parameters()
print(f"Trainable parameters: {actual_params:,}")

if lora_base.config.n_layer == 12 and lora_base.config.n_embd == 768:
    assert actual_params == 294_912, (
        f"Expected 294,912 trainable parameters; got {actual_params:,}"
    )

# ------------------------------------------------------------
# 5. TRAINING ARGUMENTS AND TRAINING FUNCTION
# ------------------------------------------------------------

def make_args(run_name, learning_rate):
    return TrainingArguments(
        output_dir=f"./results_{run_name}",
        learning_rate=learning_rate,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
        use_cpu=not torch.cuda.is_available(),
        remove_unused_columns=False,
    )

def train(model, args, train_dataset, val_dataset, run_name):
    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collator,
    )

    trainer.train()
    evaluation = trainer.evaluate()

    train_losses = [
        entry["loss"]
        for entry in trainer.state.log_history
        if "loss" in entry and "eval_loss" not in entry
    ]

    stats = {
        "train loss": train_losses[-1] if train_losses else float("nan"),
        "eval loss": evaluation.get("eval_loss", float("nan")),
    }

    return stats, evaluation.get("eval_loss", float("nan"))

# ------------------------------------------------------------
# 6. TRAIN LoRA
# ------------------------------------------------------------

lora_stats, lora_val_loss = train(
    lora_model,
    make_args("lora", LORA_LR),
    train_tokens,
    val_tokens,
    "LoRA",
)

# ------------------------------------------------------------
# 7. COMPARE WITH FULL FINE-TUNING, IF AVAILABLE
# ------------------------------------------------------------

print("\nLoRA results:")
display(pd.DataFrame([{
    **lora_stats,
    "val loss": lora_val_loss,
}], index=["LoRA"]))

if "full_stats" in globals() and "full_val_loss" in globals():
    comparison = pd.DataFrame({
        "Full fine-tuning": {
            **full_stats,
            "val loss": full_val_loss,
        },
        "LoRA": {
            **lora_stats,
            "val loss": lora_val_loss,
        },
    }).T

    print("\nTraining comparison:")
    display(comparison)
else:
    print(
        "\nFull fine-tuning results are not available in this session. "
        "Run the full fine-tuning cell to compare both methods."
    )


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Training columns: ['input_ids', 'attention_mask']
Validation columns: ['input_ids', 'attention_mask']


...

Epoch,Training Loss,Validation Loss
1,1.449686,1.436659
2,1.512307,1.415365
3,1.462323,1.407910


Training Loss,Validation Loss,Epoch
1.462323,1.407910,3



LoRA results:


,train loss,eval loss,val loss
LoRA,1.462323,1.40791,1.40791



Full fine-tuning results are not available in this session. Run the full fine-tuning cell to compare both methods.


**Question 6 — Answer guide**

(a) Each of 12 blocks adds `r × in + out × r = 8×768 + 2304×8 = 24,576`; total = `12×24,576 = 294,912`, about **0.237%** of GPT-2's 124,439,808 parameters.

(b) Copy measured time and memory from the table. LoRA still stores and runs the frozen base weights and needs activations/gradient flow through the network, so memory and runtime do not scale with trainable-parameter count alone.

(c) Compare observed validation losses **[full: ] [LoRA: ]**. Lower loss means better held-out next-token prediction here, not necessarily better recipe quality.

## Exercise 7 — Compare models and forgetting

In [20]:

# ============================================================
# CREATE THE MISSING FULL FINE-TUNED MODEL
# ============================================================

import torch
import transformers
from transformers import (
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

# Check required variables
required = ["CKPT", "DEVICE", "SEED", "tokenizer", "train_data", "val_data"]

missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(
        "Run the notebook setup/data-preparation cells first. Missing: "
        + ", ".join(missing)
    )

transformers.set_seed(SEED)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Convert raw recipe datasets to tokenized datasets if necessary
def prepare_dataset(dataset):
    if "input_ids" in dataset.column_names:
        return dataset

    def recipe_text(example):
        parts = []
        for column in ["prompt", "title", "ingredients", "directions"]:
            value = example.get(column)
            if value is not None and str(value).strip():
                parts.append(f"{column.upper()}:\n{value}")
        return {"text": "\n\n".join(parts)}

    dataset = dataset.map(recipe_text)

    dataset = dataset.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            max_length=256,
        ),
        batched=True,
    )

    keep = [
        c for c in ["input_ids", "attention_mask", "token_type_ids"]
        if c in dataset.column_names
    ]
    return dataset.remove_columns(
        [c for c in dataset.column_names if c not in keep]
    )

full_train_data = prepare_dataset(train_data)
full_val_data = prepare_dataset(val_data)

# Load a fresh model WITHOUT LoRA
full_model = AutoModelForCausalLM.from_pretrained(CKPT)
full_model.config.use_cache = False
full_model.config.pad_token_id = tokenizer.pad_token_id
full_model = full_model.to(DEVICE)

# Configure full fine-tuning
full_args = TrainingArguments(
    output_dir="./results_full_finetuning",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
    remove_unused_columns=False,
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

full_trainer = Trainer(
    model=full_model,
    args=full_args,
    train_dataset=full_train_data,
    eval_dataset=full_val_data,
    data_collator=collator,
)

# Train all model parameters
full_trainer.train()

# Evaluate and save results in variables expected by the comparison
full_eval = full_trainer.evaluate()
full_val_loss = full_eval["eval_loss"]

train_losses = [
    entry["loss"]
    for entry in full_trainer.state.log_history
    if "loss" in entry and "eval_loss" not in entry
]

full_stats = {
    "train loss": train_losses[-1] if train_losses else float("nan"),
    "eval loss": full_val_loss,
}

print("\nFull fine-tuning completed.")
print("Validation loss:", full_val_loss)
print("Full model is now available as full_model.")


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,1.222531,1.205081
2,1.210863,1.181232
3,1.114527,1.174512


Training Loss,Validation Loss,Epoch
1.114527,1.174512,3



Full fine-tuning completed.
Validation loss: 1.1745115518569946
Full model is now available as full_model.


**Question 7 — Answer guide**

(a) Summarize the actual comparison table for BLEU, ROUGE-1/2/L, ingredient coverage, average words, and stopped percentage: **[use observed values]**.

(b) A repeated house-style opening and serving suggestion can produce shared n-grams even if the central instructions differ. The metrics reward lexical overlap, not meaning or safety.

(c) Inspect the three printed examples for ingredient use, sensible ordering, invented ingredients, and unsafe instructions. Record specific observations from your outputs: **[fill in]**. The coverage heuristic only checks ingredient-name last words and cannot detect synonyms or correct use.

(d) Higher general-English loss/perplexity indicates more forgetting on these four non-cooking texts. Full fine-tuning changes all weights; LoRA changes a small adapter, though its active adapter can still alter behavior.

## Exercise 8 — Save and merge adapter

In [22]:

# ============================================================
# SAVE LoRA ADAPTER AND VERIFY MERGED MODEL
# ============================================================

import torch
from pathlib import Path

# 1. Save the adapter before merging
ADAPTER_DIR = Path(ADAPTER_DIR) if "ADAPTER_DIR" in globals() else Path("./gpt2_lora_adapter")
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

lora_model.eval()
lora_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

adapter_mb = sum(
    f.stat().st_size
    for f in ADAPTER_DIR.iterdir()
    if f.is_file()
) / 1e6

print(f"LoRA adapter saved: {ADAPTER_DIR}")
print(f"Adapter size: {adapter_mb:.2f} MB")
print(f"GPT-2 FP32 reference size: {124_439_808 * 4 / 1e6:.0f} MB")

# 2. Prepare a test input
probe = tokenizer(
    format_prompt(eval_rows[0]),
    return_tensors="pt",
).to(DEVICE)

# 3. Get logits before merging
with torch.no_grad():
    before = lora_model(**probe).logits.float().cpu()

# 4. Merge the LoRA weights into the base model
merged = lora_model.merge_and_unload()
merged.eval()

# 5. Get logits after merging
with torch.no_grad():
    merged_probe = {k: v.to(next(merged.parameters()).device)
                    for k, v in probe.items()}
    after = merged(**merged_probe).logits.float().cpu()

# 6. Report numerical differences
difference = (before - after).abs()
max_change = difference.max().item()
mean_change = difference.mean().item()

print(f"\nLargest logit difference: {max_change:.6e}")
print(f"Mean absolute logit difference: {mean_change:.6e}")

# 7. Check model structure
merged_params = sum(p.numel() for p in merged.parameters())
has_lora_parameters = any(
    "lora" in name.lower()
    for name, _ in merged.named_parameters()
)

print(f"Merged model parameters: {merged_params:,}")
print(f"LoRA parameters remain: {has_lora_parameters}")

assert not has_lora_parameters, (
    "LoRA parameters are still present in the merged model."
)

print("\nMerge operation completed and structural check passed.")
print("Review the logit differences above to assess numerical agreement.")


LoRA adapter saved: recipe_lab_outputs/adapter
Adapter size: 3.56 MB
GPT-2 FP32 reference size: 498 MB

Largest logit difference: 2.198486e-01
Mean absolute logit difference: 2.960636e-02
Merged model parameters: 124,439,808
LoRA parameters remain: False

Merge operation completed and structural check passed.
Review the logit differences above to assess numerical agreement.


**Question 8 — Answer**

(a) The adapter has 294,912 parameters, only about 0.237% of the base parameter count. Parameter storage alone at fp32 is about 1.18 MB; the saved adapter size printed above includes serialization details.

(b) Merging adds the low-rank update into weights with floating-point arithmetic, so rounding and operation order can create tiny logit differences.

(c) One base plus swappable adapters reduces duplicated base storage and supports dynamic style selection. Five merged copies can simplify serving and avoid adapter-switch overhead but consume more storage and possibly RAM/VRAM.

## Reflection (about 150 words)

**Sample reflection to finalize after your run:**

I would choose **[full fine-tuning / LoRA]** for the app based on the measured metrics, output quality, forgetting, training time, memory, and adapter storage. The metric table indicates **[summarize measured BLEU/ROUGE/coverage and length]**, while reading the generated recipes showed **[specific quality or safety observation]**. Full fine-tuning updates all GPT-2 parameters and may fit the dataset's wording more strongly, but it costs more trainable-state memory and may cause greater forgetting. LoRA trains only 294,912 parameters (about 0.237% of GPT-2), stores a small adapter, and allows one base model to serve multiple styles. However, parameter efficiency alone does not prove recipe quality or safety, and lower validation loss does not guarantee sensible steps. This experiment also uses one dataset, a small evaluation sample, one seed, and a crude ingredient-coverage heuristic. My next experiment would evaluate more held-out dishes with human ratings for ingredient fidelity, step order, and food safety, and compare several seeds and LoRA ranks under the same compute budget.